In [11]:
"""
Project: Developing ETL Pipeline for Motor Vehicle Collision Risk Analysis
Overview: Preprocessing MVC Data into an Analytical Pipeline
Author: Josue Aguilar
"""
import pandas as pd

In [12]:
# Source is motor vehicle collision dataset and by the end of preprocessing
# A new file will be prepared for an SQL database
source  = "../data/MVC.xlsx"
destination = "../data/MVC_clean.csv"

In [13]:
# Loading Data and preventing auto-formatting 
df = pd.read_excel(source, dtype=str)
print(f"{len(df):,} rows loaded")
original_len = len(df)

1,048,575 rows loaded


In [14]:
# Verification of data types and counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 25 columns):
 #   Column                       Non-Null Count    Dtype 
---  ------                       --------------    ----- 
 0   UNIQUE_ID                    1048575 non-null  object
 1   COLLISION_ID                 1048575 non-null  object
 2   CRASH_DATE                   1048575 non-null  object
 3   CRASH_TIME                   1048575 non-null  object
 4   VEHICLE_ID                   1048575 non-null  object
 5   STATE_REGISTRATION           1034684 non-null  object
 6   VEHICLE_TYPE                 1033226 non-null  object
 7   VEHICLE_MAKE                 189182 non-null   object
 8   VEHICLE_MODEL                27945 non-null    object
 9   VEHICLE_YEAR                 187761 non-null   object
 10  TRAVEL_DIRECTION             200435 non-null   object
 11  VEHICLE_OCCUPANTS            194283 non-null   object
 12  DRIVER_SEX                   166003 non-null   object
 1

In [15]:
# Standardization of column names for SQL consistency
df.columns = df.columns.str.lower()

## Crash Date and Time Cleaning

Standardizing date and time formats for SQL compatability

In [16]:
# Standardization of date and time format
df["crash_date"] = pd.to_datetime(df["crash_date"], errors="coerce")
# Filters out years past 2017 
df["crash_date"] = df["crash_date"].where(df["crash_date"].dt.year <= 2017)
df["crash_date"] = df["crash_date"].dt.strftime("%Y-%m-%d").where(df["crash_date"].notna(), None)

df["crash_time"] = pd.to_datetime(df["crash_time"], format="%H:%M:%S", errors="coerce").dt.strftime("%H:%M:%S")

In [17]:
# Removing missing data to maintain SQL integrity
df = df.dropna(subset=["crash_date", "crash_time"])

## ID Columns Cleaning

Converting ID Columns to integer datatypes and removed nulls for SQL key requirements

In [18]:
# Vehicle ID processing is completed in the schema. Currently Vehicle ID is a mixture of UUIDs and 
# identifers depending on the collision ID
df["vehicle_id"].head(10)

0          1
2          2
3          1
4          1
5     219456
7     672828
8          1
9          2
10    554272
11         2
Name: vehicle_id, dtype: object

In [19]:
# Conversion of IDs to numeric
df["collision_id"] = pd.to_numeric(df["collision_id"], errors="coerce")
df["unique_id"] = pd.to_numeric(df["unique_id"], errors="coerce")

In [20]:
# Ensure keys are prepared for integration into SQL and table constraints
before = len(df)
df = df.dropna(subset=["collision_id", "unique_id"])
df["collision_id"] = df["collision_id"].astype(int)
df["unique_id"] = df["unique_id"].astype(int)

# Verification of data quality
print(f"Dropped {before - len(df)} rows with invalid ID columns.")

Dropped 0 rows with invalid ID columns.


In [21]:
# Ensure unique identifier is actually unique
df["unique_id"].duplicated().sum()

np.int64(0)

## Travel Direction Column Cleaning

In [22]:
df["travel_direction"].unique()

array([nan, 'East', 'Southwest', 'South', 'West', 'North', 'Southeast',
       'Unknown', 'Northeast', 'Northwest', '-'], dtype=object)

In [23]:
# Standardizing values in travel direction column by mapping
dir_map = {
    "N": "North", 
    "S": "South", 
    "E": "East",  
    "W": "West",
    "NE": "Northeast", 
    "NW": "Northwest", 
    "SE": "Southeast", 
    "SW": "Southwest",
}

# Set created to group null values
dir_null = {"-", "U", "Unknown", ""}

In [24]:
def clean_direction(direction):
    """
    Standardizes values in travel direction column while also handling nulls
    """
    if pd.isna(direction):
        return None
    stripped_dir = direction.strip()
    if stripped_dir in dir_null:
        return None

    # Returns mapped value, valid title-cased name or it will return None
    return dir_map.get(stripped_dir.upper(), stripped_dir.title() if stripped_dir.upper() in
                       {value.upper() for value in dir_map.values()} else None)

In [25]:
# Applys cleaning function and verifies values found in column
df["travel_direction"] = df["travel_direction"].apply(clean_direction)
df["travel_direction"].unique()

array([None, 'East', 'Southwest', 'South', 'West', 'North', 'Southeast',
       'Northeast', 'Northwest'], dtype=object)

## Cleaning Driver Sex Column

Validation of data integrity for Driver Sex column

In [26]:
df["driver_sex"].unique()

array([nan, 'M', 'F', 'U'], dtype=object)

In [27]:
# Sex has to coincide with either M or F if not values are coerced to None
df["driver_sex"] = df["driver_sex"].apply(
    lambda x: x.strip() if pd.notna(x) and x.strip() in ("M", "F") else None
)

In [28]:
df["driver_sex"].unique()

array([None, 'M', 'F'], dtype=object)

## Cleaning Public Property Damage Column

Conversion from Y and N into binary integers for direct calculations in SQL and performance optimization when indexing.

In [29]:
df["public_property_damage"].unique()

array([nan, 'N', 'Y', 'Unspecified'], dtype=object)

In [30]:
# Map creation where Y = 1 and N = 0
ppd_map = {
    "Y": 1,
    "N": 0
}

# Values are mapped and other strings are set to None
df["public_property_damage"] = df["public_property_damage"].apply(
    lambda x: ppd_map.get(str(x).strip()) if pd.notna(x) else None
)

In [31]:
df["public_property_damage"].unique()

array([nan,  0.,  1.])

## Cleaning Contributing Factor Columns

In [32]:
sorted(df["contributing_factor_1"].dropna().unique())

['1',
 '80',
 'Accelerator Defective',
 'Aggressive Driving/Road Rage',
 'Alcohol Involvement',
 'Animals Action',
 'Backing Unsafely',
 'Brakes Defective',
 'Cell Phone (hand-Held)',
 'Cell Phone (hand-held)',
 'Cell Phone (hands-free)',
 'Driver Inattention/Distraction',
 'Driver Inexperience',
 'Driverless/Runaway Vehicle',
 'Drugs (Illegal)',
 'Drugs (illegal)',
 'Eating or Drinking',
 'Failure to Keep Right',
 'Failure to Yield Right-of-Way',
 'Fatigued/Drowsy',
 'Fell Asleep',
 'Following Too Closely',
 'Glare',
 'Headlights Defective',
 'Illnes',
 'Illness',
 'Lane Marking Improper/Inadequate',
 'Listening/Using Headphones',
 'Lost Consciousness',
 'Obstruction/Debris',
 'Other Electronic Device',
 'Other Lighting Defects',
 'Other Vehicular',
 'Outside Car Distraction',
 'Oversized Vehicle',
 'Passenger Distraction',
 'Passing Too Closely',
 'Passing or Lane Usage Improper',
 'Pavement Defective',
 'Pavement Slippery',
 'Pedestrian/Bicyclist/Other Pedestrian Error/Confusion',
 

In [33]:
sorted(df["contributing_factor_2"].dropna().unique())

['1',
 'Accelerator Defective',
 'Aggressive Driving/Road Rage',
 'Alcohol Involvement',
 'Animals Action',
 'Backing Unsafely',
 'Brakes Defective',
 'Cell Phone (hand-Held)',
 'Cell Phone (hands-free)',
 'Driver Inattention/Distraction',
 'Driver Inexperience',
 'Driverless/Runaway Vehicle',
 'Drugs (illegal)',
 'Eating or Drinking',
 'Failure to Keep Right',
 'Failure to Yield Right-of-Way',
 'Fatigued/Drowsy',
 'Fell Asleep',
 'Following Too Closely',
 'Glare',
 'Headlights Defective',
 'Illnes',
 'Lane Marking Improper/Inadequate',
 'Listening/Using Headphones',
 'Lost Consciousness',
 'Obstruction/Debris',
 'Other Electronic Device',
 'Other Lighting Defects',
 'Other Vehicular',
 'Outside Car Distraction',
 'Oversized Vehicle',
 'Passenger Distraction',
 'Passing Too Closely',
 'Passing or Lane Usage Improper',
 'Pavement Defective',
 'Pavement Slippery',
 'Pedestrian/Bicyclist/Other Pedestrian Error/Confusion',
 'Physical Disability',
 'Prescription Medication',
 'Reaction to U

In [34]:
df["contributing_factor_1"].value_counts()

contributing_factor_1
Unspecified                       676004
Driver Inattention/Distraction     96495
Fatigued/Drowsy                    32318
Failure to Yield Right-of-Way      30186
Other Vehicular                    28521
                                   ...  
Vehicle Vandalism                     14
1                                     13
Eating or Drinking                    13
Texting                                8
Listening/Using Headphones             1
Name: count, Length: 61, dtype: int64

In [35]:
# Map to standardize contributing factor labels
cf_map = {
    "cell phone (hand-held)":   "Cell Phone (hand-held)",
    "cell phone (hand-Held)":   "Cell Phone (hand-held)",
    "drugs (illegal)":          "Drugs (Illegal)",
    "illnes":                   "Illness",
    "reaction to other uninvolved vehicle": "Reaction to Uninvolved Vehicle",
}

# Removes unusable values from the list of factors
cf_null = {"unspecified", "1", "80"}

In [36]:
def clean_cf(cf):
    """
    Standardizes contributing factors using tiered mapping while obtaining
    None for noise
    """
    if pd.isna(cf):
        return None
    stripped_cf = cf.strip()
    if stripped_cf.lower() in cf_null:
        return None
    return cf_map.get(stripped_cf, cf_map.get(stripped_cf.lower(), stripped_cf))

In [37]:
# Application of standardization function
df["contributing_factor_1"] = df["contributing_factor_1"].apply(clean_cf)
df["contributing_factor_2"] = df["contributing_factor_2"].apply(clean_cf)

In [38]:
# Dropping rows where contributing factor is not documented. Focus for this analysis
# will be on active accidents. Missing values are often attributed to parked or passive accidents.
before = len(df)
df = df.dropna(subset=["contributing_factor_1"])
print(f"Dropped {before - len(df)} rows with no contributing factors")

Dropped 693781 rows with no contributing factors


## Cleaning Vehicle Types column

Standardizing vehicle types column by manual mapping for car types that appear most frequently.

In [39]:
# Cleaning Vehicle Types
sorted(df["vehicle_type"].dropna().unique())

['00',
 "12' o",
 '2 dr sedan',
 '2000',
 '250-3',
 '26 ft',
 '2dr',
 '3-Door',
 '4 dr sedan',
 '4door',
 '985',
 'AM/TR',
 'AMB',
 'AMBU',
 'AMBUL',
 'AMBULANCE',
 'APPOR',
 'Ambul',
 'Ambulance',
 'Armored Truck',
 'BACK',
 'BACKH',
 'BICYCLE',
 'BKHOE',
 'BOBCA',
 'BOX H',
 'BOX T',
 'BUDGE',
 'BUS',
 'BUSS',
 'Beverage Truck',
 'Bike',
 'Box T',
 'Box Truck',
 'Budge',
 'Bulk Agriculture',
 'Bus',
 'C1',
 'CAR T',
 'CARGO',
 'CART',
 'CASE',
 'CAT',
 'CATER',
 'CITY',
 'COACH',
 'COLL',
 'COM',
 'COMME',
 'CON E',
 'CONTR',
 'CRANE',
 'Carry All',
 'Chassis Cab',
 'Cmix',
 'Comme',
 'Concrete Mixer',
 'Convertible',
 'DELIV',
 'DELV',
 'DIRT',
 'DODGE',
 'DUMP',
 'DUMPS',
 'DUMPT',
 'Delie',
 'Deliv',
 'Dump',
 'E BIK',
 'E-BIK',
 'E1',
 'EBIKE',
 'ECOM',
 'ECONO',
 'ELECT',
 'ENGIN',
 'EXCAV',
 'Ebike',
 'Econo',
 'Elect',
 'FDNY',
 'FED',
 'FIRE',
 'FIRE TRUCK',
 'FIRER',
 'FIRET',
 'FLAT',
 'FLATB',
 'FLTRL',
 'FOOD',
 'FORD',
 'FORK',
 'FORKL',
 'FREIG',
 'FREIH',
 'Fire',
 'Fi

In [40]:
# Investigate top vehicle types
df["vehicle_type"].value_counts().head(20)

vehicle_type
PASSENGER VEHICLE                      123683
SPORT UTILITY / STATION WAGON           58640
Station Wagon/Sport Utility Vehicle     26443
Sedan                                   24479
4 dr sedan                              17457
TAXI                                    14952
UNKNOWN                                 12230
VAN                                     10027
OTHER                                    7205
LARGE COM VEH(6 OR MORE TIRES)           6839
SMALL COM VEH(4 TIRES)                   5892
BUS                                      4524
LIVERY VEHICLE                           4509
Taxi                                     4446
PICK-UP TRUCK                            4399
BICYCLE                                  3290
Pick-up Truck                            2742
Box Truck                                2569
Bike                                     1727
Bus                                      1663
Name: count, dtype: int64

In [41]:
# Prioritizes mapping on vehicle types that appear most frequently
# for cleaner aggregation and visualization
vehtype_map = {
    "sedan":                                "Sedan",
    "4 dr sedan":                           "Sedan",
    "2 dr sedan":                           "Sedan",
    "4dr":                                  "Sedan",
    "2dr":                                  "Sedan",
    "station wagon/sport utility vehicle":  "Sport Utility Vehicle",
    "sport utility / station wagon":        "Sport Utility Vehicle",
    "pick-up truck":                        "Pick-up Truck",
    "pk":                                   "Pick-up Truck",
    "taxi":                                 "Taxi",
    "bus":                                  "Bus",
    "motorcycle":                           "Motorcycle",
    "bike":                                 "Bicycle",
    "van":                                  "Van",
    "unknown":                              None,
    "other":                                None,
}


# Set to help remove null values
vehtype_null = {"unknown","other","unk","unkno","unkow"}                             

In [42]:
def clean_vehtype(vehicle):
    """
    Vehicle types standardization while filtering non-descriptive data.
    Ensures title-case for accurate vehicle types or None for null values
    """
    if pd.isna(vehicle):
        return None
    stripped_veh = str(vehicle).strip().lower()
    
    if stripped_veh in vehtype_null:
        return None
        
    if stripped_veh in vehtype_map:
        return vehtype_map[stripped_veh]
    # Vehicle types are typically not less than 2 letters and excldues ambigous numeric codes
    if len(stripped_veh) >= 3 and not stripped_veh.isnumeric():
        return stripped_veh.title()
    return None

In [43]:
# Applies mapping function
df["vehicle_type"] = df["vehicle_type"].apply(clean_vehtype)

## Cleaning Vehicle Year column

Vehicle year had several values that were just not possible and had to be cleaned. Column also needed to at least be a numeric type before being moved into SQL

In [44]:
df["vehicle_year"].unique()

array([nan, '2015', '2005', '2000', '1993', '1994', '2012', '2003',
       '2016', '2006', '1998', '2008', '2014', '1989', '2013', '1995',
       '2004', '2010', '2011', '2007', '2009', '2002', '2001', '2017',
       '1999', '1988', '1986', '1997', '1996', '1969', '1971', '2019',
       '1991', '2100', '1992', '2018', '1990', '9999', '2022', '1978',
       '2816', '1985', '2096', '8014', '1980', '1983', '2050', '1984',
       '1987', '1964', '2106', '1972', '2040', '1981', '2993', '2033',
       '2110', '6666', '1951', '1949', '1979', '5004', '2915', '1960',
       '2045', '2815', '2020', '1977', '1970', '1958', '2024', '2051',
       '1956', '1982', '1900', '2207', '2104', '2916', '1968', '2046',
       '2914', '1963', '3015', '1965', '8018', '1975', '1976', '2083',
       '19969', '2913', '2994', '1966', '2208', '2201', '2101', '5016',
       '2041', '2023', '1936', '4998', '1000'], dtype=object)

In [45]:
# Converting year to numeric, filtering for realistic range and cast to int
df["vehicle_year"] = pd.to_numeric(df["vehicle_year"], errors="coerce")
df["vehicle_year"] = df["vehicle_year"].where(df["vehicle_year"].between(1900, 2017))

# Final integer typing will happen in SQL
df["vehicle_year"] = df["vehicle_year"].apply(lambda x: int(x) if pd.notna(x) else None)

# Checking validity of results
sorted(df["vehicle_year"].dropna().unique(), reverse = True)

[np.float64(2017.0),
 np.float64(2016.0),
 np.float64(2015.0),
 np.float64(2014.0),
 np.float64(2013.0),
 np.float64(2012.0),
 np.float64(2011.0),
 np.float64(2010.0),
 np.float64(2009.0),
 np.float64(2008.0),
 np.float64(2007.0),
 np.float64(2006.0),
 np.float64(2005.0),
 np.float64(2004.0),
 np.float64(2003.0),
 np.float64(2002.0),
 np.float64(2001.0),
 np.float64(2000.0),
 np.float64(1999.0),
 np.float64(1998.0),
 np.float64(1997.0),
 np.float64(1996.0),
 np.float64(1995.0),
 np.float64(1994.0),
 np.float64(1993.0),
 np.float64(1992.0),
 np.float64(1991.0),
 np.float64(1990.0),
 np.float64(1989.0),
 np.float64(1988.0),
 np.float64(1987.0),
 np.float64(1986.0),
 np.float64(1985.0),
 np.float64(1984.0),
 np.float64(1983.0),
 np.float64(1982.0),
 np.float64(1981.0),
 np.float64(1980.0),
 np.float64(1979.0),
 np.float64(1978.0),
 np.float64(1977.0),
 np.float64(1976.0),
 np.float64(1975.0),
 np.float64(1972.0),
 np.float64(1971.0),
 np.float64(1970.0),
 np.float64(1969.0),
 np.float64(1

## Cleaning Vehicle Occupants Column

Standardizing vehicle occupants to a realistic number and converting the column to numeric.

In [46]:
# Converting occupants to numeric, filtering for realistic range and cast to int
df["vehicle_occupants"] = pd.to_numeric(df["vehicle_occupants"], errors="coerce")
df["vehicle_occupants"] = df["vehicle_occupants"].where(df["vehicle_occupants"].between(0, 100))
df["vehicle_occupants"] = df["vehicle_occupants"].apply(lambda x: int(x) if pd.notna(x) else None)
df["vehicle_occupants"].max()

np.float64(62.0)

## General Cleaning for Remaining Columns

In [47]:
# Removing invalid values from remaining columns to None for consistent null values throughout the analysis
null_strings = {"nan", "none", "unspecified", "unknown", "n/a", ""}
str_cols = [
    "state_registration", "driver_license_status", "driver_license_jurisdiction",
    "pre_crash", "point_of_impact", "vehicle_damage",
]
for col in str_cols:
    df[col] = df[col].apply(
        lambda x: x.strip() if pd.notna(x) and str(x).strip().lower() not in null_strings else None
    )


In [48]:
# Writing results into new csv to be used for final preparations before SQL
df.to_csv(destination, index=False)